# Messy Crime Incident Dataset for Data Cleaning Practice

## Goal and description of this script 

This dataset contains simulated crime incidents reported across different districts and cities, designed specifically for **data cleaning and preprocessing practice**. Each row corresponds to a single incident with details about the crime, suspect information, reporting method, and investigation status. The data intentionally includes inconsistencies, missing values, duplicates, and formatting issues to mimic real-world messy datasets.

# I) Data Cleaning
## Removal of non informative rows
I load my dataset and convert it into a table. Then I remove the rows without values (NaN) and duplicates. I then remove spaces and convert to lowercase to avoid creating new categories only due to case-sensitvity.

In [ ]:
# Import libraries and read in the dataset
import pandas as pd
from pathlib import Path

dataset = Path(
    r"C:\Users\EMI-M\OneDrive\Lauriane\Informatique\Self_training - Portfolio\Kaggle data _ clean\crime_incidents_messy.csv"
)

df = pd.read_csv(
    dataset,
    sep=",",
    encoding="utf-8-sig",
    skipinitialspace=True,
    on_bad_lines="warn"
)

# Inspect the dataset : column names, first rows, column types, missing values
print(df.head(5))
df.info() 
print(df.isna().sum())

# Check empty rows and duplicates
print("Fully empty rows:", df.isna().all(axis=1).sum())
print("Duplicate rows:", df.duplicated().sum())

# Remove fully empty rows and duplicates
df = df.dropna(how="all")
df = df.drop_duplicates()

print("Shape after cleaning:", df.shape)

# Rename columns to make them more readable
##  Example: crime_type -> Crime type
df.columns = (
    df.columns
      .str.replace("_", " ", regex=False)
      .str.capitalize())

# Standardize string columns
cat_cols = df.select_dtypes(["object", "string"]).columns

# Remove spaces at the beginning and end
df[cat_cols] = df[cat_cols].apply(
    lambda s: s.str.strip())

# Convert text to lowercase
df[cat_cols] = df[cat_cols].apply(
    lambda s: s.str.lower())

# Replace multiple spaces with one space
df[cat_cols] = df[cat_cols].apply(
    lambda s: s.str.replace(r"\s+", " ", regex=True))

: 

## Datatype correction

I corrected the datatype of `Badge number` and `Property loss usd`.
I noticed that some property loss values had an extra ".0", creating additional NaN values. I removed the extra '.0' before the conversion, so these values are kept.

In [ ]:
# change badge number datatype to string, but keep NaN values as <NA>
df["Badge number"] = df["Badge number"].astype("Int64").astype("string")

# convert property loss to numeric; non-numeric values become NaN instead of raising an error
df["Property loss usd"] = pd.to_numeric(
    df["Property loss usd"].str.replace(r"^(\d+\.\d+)\.0$", r"\1", regex=True),
    errors="coerce")

## Cleaning of categorical values

### Typo correction and category harmonization

Typos create extra categories: the same value spelled differently is counted as two distinct categories. Identifying and correcting them is essential to avoid these duplicates. With more than 5,000 rows, a manual check would take too long, so I automated the detection of close matches. I then reviewed each proposed correction and manually adjusted the ones that were wrong.

In [ ]:
# Look at the values in the categorical columns to identify potential typos or inconsistencies

for c in cat_cols:
    print(f"\n{c} ({df[c].nunique()} unique)")
    print(df[c].value_counts(dropna=False).head(5).to_string())

import difflib

# Create a mapping of old values to new values
mappings = {}
for c in cat_cols:
    counts = df[c].value_counts()
    if len(counts) > 100:
        continue
    kept, mapping = [], {}
    for v in counts.index:
        match = difflib.get_close_matches(v, kept, n=1, cutoff=0.85)
        if match:
            mapping[v] = match[0]
        else:
            kept.append(v)
    if mapping:
        mappings[c] = mapping

for c, mapping in mappings.items():
    counts = df[c].value_counts()
    print(f"\n{c}")
    for old, new in mapping.items():
        print(f"  {old!r} ({counts[old]}) → {new!r} ({counts[new]})")

# Remove the mappings that are not correct    
del mappings["District"]["northwest"] 
del mappings["District"]["southeast"]

# fix reversed direction (the typo was more frequent)
mappings["Case status"] = {"pendng": "pending"}

# apply changes 
for c, mapping in mappings.items():
    df[c] = df[c].replace(mapping)

print ("\n")

# check it worked
for c, mapping in mappings.items():
    left = df[c].isin(mapping.keys()).sum()
    print(c, "remaining old values:", left)


### Harmonizing category synonyms

The similarity check above cannot catch every duplicate category: it misses abbreviations and synonyms, whose spelling is too different. For example, "sa" stands for "sexual assault", and "narcotics" means the same as "drug offense". This step cannot be automated and had to be done manually.


In [ ]:
cols = ["District", "Crime type", "Weapon used", "Severity", "Case status", "Resolution", "Reported online"]

unique_table = pd.DataFrame(
    {c: pd.Series(sorted(df[c].dropna().unique(), key=str)) for c in cols}
).fillna("")
print(unique_table)

#group synonyms
synonyms = {
    "District": {
        "central": ["cen"],
        "midtown": ["mid"],
        "north": ["nor"],
        "south": ["sou"],
    },
    "Crime type": {

    # Crimes against persons
    "person-related": [
        "homicide", "murder", "manslaughter",
        "assault", "asslt", "battery", "assault & battery",
        "sexual assault", "sa",
        "kidnapping", "abduction",
        "domestic violence", "dom. violence", "dv"
    ],

    # Property crimes
    "property": [
        "theft", "theft/larceny", "larceny", "stealing",
        "burglary", "b&e", "breaking & entering",
        "robbery", "armed robbery",
        "arson", "arsen", "fire setting",
        "vandalism", "graffiti", "property damage"
    ],

    # Financial / cyber crimes
    "financial / cyber": [
        "fraud", "scam", "deception", "fraudulent activity",
        "cyber crime", "hacking", "online fraud"
    ],

    # Public order / society
    "public order / society": [
        "drug offense", "drugs", "narcotics",
        "dui", "d.u.i.", "dwi", "drunk driving",
        "trespassing", "trespass"]
    },
    "Severity": {  
          "1" : ["low"],
          "2" : ["medium", "med"],
          "3" : ["high"],
          "4" : ["critical", "crit"]
    },
    "Case status": {
        "under investigation": ["investgation"],
    },
    "Resolution": {
        "warning": ["warning issued"],
        "dismissed": ["case dismissed"],
    },
    "Weapon used": {
       "firearm" : ["gun", "rifle", "pistol"],
        "unarmed" : ["hands/feet", "hands"],
        "blunt object" : ["bat"]
    },
    "Reported online": {
        "yes": ["1", "true"],
        "no": ["0", "false"]    
}}

for col, groups in synonyms.items():
    mapping = {old: new for new, olds in groups.items() for old in olds}
    df[col] = df[col].replace(mapping)
    print(col, df[col].nunique(), sorted(df[col].dropna().unique()))

# change unknown category into NaN
df["Suspect race"] = (
    df["Suspect race"]
    .astype("string")
    .replace("unknown", pd.NA)
)

### Missing gender values

Missing values are always a challenge because we want to avoid losing information. For the victim and suspect gender columns, the missing values can be imputed from the first name columns. However, with more than 5,000 rows, filling them manually would be too time-consuming. Fortunately, a package (`gender-guesser`) can guess gender from first names. It also returns a category for ambiguous names ("mostly male" or "mostly female"). I reviewed this category and decided which names were truly ambiguous. All imputed values are flagged.

In [ ]:
# Handling missing data : victim and suspect gender
import gender_guesser.detector as gender

d = gender.Detector(case_sensitive=False)
labels = {"male": "male", "mostly_male": "male",
          "female": "female", "mostly_female": "female"}

for role in ["Victim", "Suspect"]:
    first, gcol = f"{role} first name", f"{role} gender"
    flag, unc = f"{role} gender imputed", f"{role} gender uncertain"
    print(f"\n===== {role} =====")

    df[gcol] = df[gcol].replace({"m": "male", "f": "female"})
    print(df[gcol].value_counts(dropna=False))

    missing = df[first].notna() & (df[gcol].isna() | (df[gcol] == "unknown"))

    # guess the gender based on the first name
    raw = df.loc[missing, first].apply(d.get_gender)
    guess = raw.map(labels).dropna()
    uncertain = raw[raw.str.startswith("mostly")].index   # "mostly_..." guesses

    for f in [flag, unc]:   # avoids resetting the flags on rerun
        if f not in df.columns:
            df[f] = False
    df.loc[guess.index, gcol] = guess
    df.loc[guess.index, flag] = True
    df.loc[uncertain, unc] = True

    print("filled:", len(guess), "/", missing.sum(), "| uncertain:", len(uncertain))
    print(df[gcol].value_counts(dropna=False))

    left = df[first].notna() & (df[gcol].isna() | (df[gcol] == "unknown"))
    print(df.loc[left, first].map(d.get_gender).value_counts())

for role in ["Victim", "Suspect"]:
    unc = f"{role} gender uncertain"
    print(f"\n===== {role} =====")
    print(df.loc[df[unc]]
            .groupby(f"{role} first name")[f"{role} gender"]
            .agg(["first", "count"])
            .to_string())
print ("\n")

# manually confirmed names: no longer uncertain
confirmed = ["mary", "carol"]

for role in ["Victim", "Suspect"]:
    unc = f"{role} gender uncertain"
    ok = df[unc] & df[f"{role} first name"].isin(confirmed)
    df.loc[ok, unc] = False
    print(role, "confirmed:", ok.sum(), "| still uncertain:", df[unc].sum())

## Cleaning of numeric values

### Data distribution overview
For each numeric column, I computed the minimum, maximum, median, mean and standard deviation, and compared them with realistic values. The first obvious problem was negative values, which are impossible for ages, number of arrests and property loss. I assumed the minus sign was an entry error and converted these values to positive (longitude was left unchanged, because US longitudes are negative).

One of the biggest questions when cleaning data is how to handle missing values: remove these rows or impute them with the median? When more than about 5% of values are missing, median imputation is generally not recommended, because it distorts the distribution. Missing value percentage for number of arrests and property loss are above 5% and values aren't imputable from other columns, Therefore I let them as they are.

In [ ]:
# Identify the numeric columns and compute summary statistics 
num_cols = df.select_dtypes("float").columns
stats = df[num_cols].agg(["min", "max", "median", "mean", "std"]).T
stats["missing_%"] = df[num_cols].isna().mean() * 100
print(stats.round(2))

# Make all numeric values positive except longitude (which can be negative in the Western Hemisphere)
for col in num_cols.drop("Longitude"):
    df[col] = df[col].abs()

# round all numeric values to 2 decimal places
df[num_cols] = df[num_cols].round(2)

### Inconsistent age

Some victim and suspect ages were unrealistically high (above 100). Before setting them to NaN, I checked whether any ages were close to 100. Since there were none, there was no risk of excluding a very old person, so all ages above 100 were set to NaN.
Since 10.7% of victim ages and 20.8% of suspect ages were missing in the original data (see table above), and even more after the invalid ages were set to NaN, I left them as NaN.

In [ ]:
import numpy as np

print("====== Victim and suspect ages ======")

# Check for outliers in the age columns
for c in ["Victim age", "Suspect age"]:
    print(c, (df[c] > 100).sum())
    print(df.loc[df[c] > 100, c].value_counts().sort_index())

print("\n")

# replace the outliers with NaN
for c in ["Victim age", "Suspect age"]:
    df.loc[df[c] > 100, c] = pd.NA


**Comment:**  A high share of missing values may also indicate that they are not missing at random. For example, some suspects may never have been identified, so personal information like their age could not be recorded.

## Inconsistent coordinates

176 latitudes were above 90, outside the valid range (−90 to 90). I set these coordinates to NaN, which brought missing values to 8.5% (latitude) and 8.7% (longitude). Even after this correction, the coordinates are not reliable: every city has almost the same median position, and positive longitudes (up to 99.8) are impossible for US cities. For location-based analyses, I therefore use the City and District columns instead.


In [ ]:

# Check for invalid coordinates (latitude: -90 to 90, longitude: -180 to 180)
bad = (df["Latitude"].notna() & ~df["Latitude"].between(-90, 90)) | \
      (df["Longitude"].notna() & ~df["Longitude"].between(-180, 180))
print("invalid coordinates:", bad.sum())
print(df.loc[bad, ["Latitude", "Longitude"]].head(10))

# Set invalid coordinates to NaN
df.loc[bad, ["Latitude", "Longitude"]] = np.nan

print("\n")

# Check the percentage of missing values
for c in ["Latitude", "Longitude"]:
    print(c, f"{df[c].isna().mean():.1%} missing")

# Generate a table of median, min, and max coordinates for each city
coord_tab = (
    df[["City", "Longitude", "Latitude"]]
    .groupby("City")
    .agg(["median", "min", "max"])
)

print(coord_tab)

## Handling date and time column 

The Incident datetime column contains the date and, in most rows, the time, so I split it into two columns: Incident date and Incident time. Dates came in three formats: 2024-04-16, 04/16/2024 (American, month first) and 16-04-2024 (day first). I converted each format explicitly so that days and months are not swapped. I then created two categories: time of day (night 0–6 h, morning 6–12 h, afternoon 12–18 h, evening 18–24 h) and season (starting on 21 March, 21 June, 21 September and 21 December).

In [ ]:
# -------------------------
# TIME
# -------------------------
# Extract time (HH:MM or HH:MM:SS)
df["Incident time"] = df["Incident datetime"].str.extract(r'([^ ]*:[^ ]*)')
print(df["Incident time"].head(5))

print("\n")

# extract hour from incident_time
hour = pd.to_numeric(df["Incident time"].str[:2], errors="coerce")   # with coerce , invalid/missing values become NaN

# Create categories
conditions = [
    (hour >= 6) & (hour < 12),
    (hour >= 12) & (hour < 18),
    (hour >= 18) & (hour < 24)
]

categories = [
    "Morning",
    "Afternoon",
    "Evening"
]

df["Time of day"] = np.select(
    conditions,
    categories,
    default="Night"
)

# ensure that NaN values aren't converted in default "night" 
df.loc[hour.isna(), "Time of day"] = pd.NA

print(df[["Incident time", "Time of day"]].head(5))

# Remove the time from the original string to get the date
df["Incident date"] = (df["Incident datetime"].str.replace(r'([^ ]*:[^ ]*)', '', regex=True).str.strip())

print("\n")

# Uniform the date format : 
df["Incident date"] = (
    pd.to_datetime(df["Incident date"], format="%Y-%m-%d", errors="coerce")
      .fillna(pd.to_datetime(df["Incident date"], format="%m/%d/%Y", errors="coerce"))
      .fillna(pd.to_datetime(df["Incident date"], format="%d-%m-%Y", errors="coerce"))
)

print(df["Incident date"].head(5))

# -------------------------
# DATE
# -------------------------
# Extract month-day as MM-DD
month_day = df["Incident date"].dt.strftime("%m-%d")

# create categories for the months
conditions = [
    (month_day >= "03-21") & (month_day < "06-21"),
    (month_day >= "06-21") & (month_day < "09-21"),
    (month_day >= "09-21") & (month_day < "12-21")
]

categories = [
    "Spring",
    "Summer",
    "Autumn",
]

df["Season"] = np.select(
    conditions,
    categories,
    default= "Winter"
)

# ensure that NaN values aren't converted in default "Winter" 
df.loc[df["Incident date"].isna(), "Season"] = pd.NA

print(df[["Incident date", "Season"]].head(5))


# II) Data visualization and exploration

### Descriptive statistics

I started the analysis with a variable overview. 
Latitude and longitude are excluded from the numeric variables. Categorical columns are manually entered due to high number of variables that I judged non relevant for the study (e.g. badge number)

In [ ]:
# expand all columns and rows when displaying the dataframe
pd.set_option("display.max_columns", None)
pd.set_option("display.expand_frame_repr", False)
pd.set_option("display.width", 200)

# split the dataset into numeric and categorical columns and compute summary statistics for each
numeric_cols = df.select_dtypes("number").columns.difference(["Latitude", "Longitude"])

numcols = df[numeric_cols].describe().round(2).T

categorical_cols = [
    "Crime type",
    "District",
    "City",
    "State",
    "Suspect gender",
    "Suspect race",
    "Victim gender",
    "Weapon used",
    "Severity",
    "Case status",
    "Resolution",
    "Reported online",
    "Season",
    "Time of day"
]

catcols = df[categorical_cols].describe().T

print (numcols)
print ("\n")
print (catcols)

### Data visualization

It is important to understand the distribution of each variable before performing further analyses. None of the numerical variables appear to follow a Gaussian distribution; instead, their distributions are relatively flat.

In [ ]:
import matplotlib.pyplot as plt
import math

# -------------------------
# CATEGORICAL COLUMNS
# -------------------------

ncols = 3
nrows = math.ceil(len(categorical_cols) / ncols)

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(15, 4 * nrows)
)

axes = axes.flatten()

for i, col in enumerate(categorical_cols):

    df[col].value_counts(dropna=False).plot(
        kind="bar",
        ax=axes[i]
    )

    axes[i].set_title(col)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Count")
    axes[i].tick_params(axis="x", rotation=45)

# Remove unused empty plots
for j in range(len(categorical_cols), len(axes)):
    axes[j].remove()

plt.tight_layout()
plt.show()


# -------------------------
# NUMERIC COLUMNS
# -------------------------

nrows = math.ceil(len(numeric_cols) / ncols)

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(15, 4 * nrows)
)

axes = axes.flatten()

for i, col in enumerate(numeric_cols):

    df[col].plot(
        kind="hist",
        bins=20,
        ax=axes[i]
    )

    axes[i].set_title(col)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Count")

# Remove unused empty plots
for j in range(len(numeric_cols), len(axes)):
    axes[j].remove()

plt.tight_layout()
plt.show()

# III) Data Analysis  
In this part, I use the cleaned data to answer a few questions. These analyses are not exhaustive: they illustrate the kind of questions this dataset can answer.

## 1. Identify crime patterns by city : Which cities report the highest number of incidents? Is there a difference in crime type depending on city?

First, I used visualizations and contingency tables to explore crime patterns across cities. Riverside had the highest number of incidents (705). Although total crime counts appeared relatively similar, a chi-square goodness-of-fit test showed that crime counts differed significantly across cities (p=0.03).
Across all cities, Property and Person-related crimes were the most frequent categories (1,653 and 1,625 incidents, respectively).
A chi-square test of independence found no significant association between city and crime type (p = 0.26), indicating that the relative distribution of crime types was similar across cities.

Conclusion: Total crime volume differed significantly across cities, but the crime-type distribution did not.

Important note: Several cities in the dataset share the same name. For simplicity, cities with identical names were treated as a single category in this analysis, regardless of state.

In [ ]:

# =============================================
# check if crime number differs between cities
# =============================================

from scipy.stats import chisquare, chi2_contingency

#  Calculate total crimes per city 
city_counts = df["City"].value_counts() 
print(city_counts)

# visualize the total crimes per city
city_counts.plot(kind="bar", figsize=(8, 4))
plt.title("Total crimes per city")
plt.xlabel("City")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# perform chi2 goodness-of-fit test
chi2, p = chisquare(city_counts)
print(f"\nGoodness-of-fit: chi2 = {chi2:.1f}, df = {len(city_counts) - 1}, p = {p:.3g}")
print ("\n")

# ============================================================
# check if crime type proportions differ between cities
# ============================================================
# create a contingency table of crime types by city
table = pd.crosstab(df["City"], df["Crime type"])

# display the contingency table with a total row (if included in the main table, it will be included in the chi2 test)
table_display = table.copy()
table_display.loc["TOTAL"] = table_display.sum(axis=0)

print(table_display)

# plot the percentages of each crime type for each city
table.div(table.sum(axis=1), axis=0).plot(kind="bar", stacked=True, figsize=(11, 5))
plt.title("Crime type mix by city")
plt.xlabel("City")
plt.ylabel("Proportion of city's crimes")
plt.xticks(rotation=45)
plt.legend(title="Crime type", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

# perform chi2 test of independence
chi2, p, dof, expected = chi2_contingency(table)
print(f"\nIndependence: chi2 = {chi2:.1f}, df = {dof}, p = {p:.3g}")
print(f"Cells with expected count < 5: {(expected < 5).sum()} of {expected.size} ")

## 2. Analyze the number of incidents over time

I analyzed variation in the number of incidents over time. First, I counted incidents per month (2018–2024) and fitted a Poisson regression to test for an overall trend: incidents increased by about 1.1 % per year, but this trend was not significant (p = 0.12). I then compared the total number of incidents by season, regardless of year: a chi-square goodness-of-fit test found no significant difference between seasons (p = 0.32). 

Conclusion: it suggests that the number of incidents was stable over time, with no clear trend or seasonal pattern

In [ ]:
import statsmodels.api as sm

# -------------------------------------------------
# 1) INCIDENTS PER MONTH
# -------------------------------------------------
incidents = df["Incident date"].dt.to_period("M").value_counts().sort_index()

# months without any incident must appear as 0, not be missing
all_months = pd.period_range(incidents.index.min(), incidents.index.max(), freq="M")
incidents = incidents.reindex(all_months, fill_value=0)

# -------------------------------------------------
# 2) POISSON REGRESSION: trend over time
# -------------------------------------------------
t = np.arange(len(incidents))  # months since start
X = sm.add_constant(t)
model = sm.GLM(incidents.values, X, family=sm.families.Poisson()).fit()
print(model.summary())

# dispersion ~1 -> Poisson is fine; clearly >1 -> use quasi-Poisson: .fit(scale="X2")
dispersion = model.pearson_chi2 / model.df_resid
irr = np.exp(model.params[1])

print(f"Dispersion: {dispersion:.2f}")
print(f"IRR per month: {irr:.4f} | per year: {irr**12:.3f} ({(irr**12 - 1) * 100:+.1f}% per year)")
print(f"p-value: {model.pvalues[1]:.3g}")

# -------------------------------------------------
# 3) PLOT: monthly incidents + fitted Poisson trend
# -------------------------------------------------
dates = incidents.index.to_timestamp()

plt.figure(figsize=(8, 4))
plt.figure(figsize=(8, 4))
plt.plot(dates, incidents.values, color="tab:blue", alpha=0.5, linewidth=1.5, label="Monthly incidents")
plt.plot(dates, model.fittedvalues, color="tab:red", linewidth=2, linestyle="--", label="Poisson trend")
plt.title("Number of incidents per month")
plt.xlabel("Date")
plt.ylabel("Number of incidents")
plt.legend()
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------------------------------------------
# 4) SEASONALITY: incidents by season
# ------------------------------------------------------------------------------------------------------

# Total number of incidents for each season
per_season = df.groupby("Season").size()

# Put seasons in chronological order
per_season = per_season.reindex(
    ["Winter", "Spring", "Summer", "Autumn"])

# Plot
per_season.plot(
    kind="bar",
    figsize=(8, 4)
)
plt.title("Total incidents per season")
plt.xlabel("Season")
plt.ylabel("Total incidents")
plt.tight_layout()
plt.show()

from scipy.stats import chisquare

chi2, p = chisquare(per_season)

print("Chi-square:", round(chi2, 2))
print("p-value:", round(p, 4))

## 3. Check whether crime severity is associated with property loss

I first tested whether property loss differed across severity grades using a Kruskal–Wallis test, as the property-loss distribution was non-normal. For this test, severity grades were treated as separate, unordered categories. I then used a Spearman rank correlation to assess whether higher severity grades were associated with higher property loss, treating severity as an ordered variable from 1 (lowest severity) to 4 (highest severity).

Property loss did not differ significantly across severity grades (p = 0.79), and there was no significant correlation between severity and property loss (p = 0.39).

Conclusion: Crime severity was not associated with property loss in this dataset.

In [ ]:
from scipy.stats import kruskal, spearmanr

# keep only rows where both values are known, so both tests use the same data
sev = df[["Severity", "Property loss usd"]].dropna().copy()
sev["Severity"] = sev["Severity"].astype(int)   # stored as text "1"-"4"

print(sev.groupby("Severity")["Property loss usd"].agg(["count", "mean", "median", "std"]).round(1))

# 1) Severity as categories: does the mean loss differ between grades?
# plot
sev.boxplot(column="Property loss usd", by="Severity", figsize=(8, 4))
plt.suptitle("")   # remove the automatic pandas title
plt.title("Property loss by severity")
plt.xlabel("Severity (1 = low, 4 = critical)")
plt.ylabel("Property loss (USD)")
plt.tight_layout()
plt.show()

# K-W test
groups = [g["Property loss usd"] for _, g in sev.groupby("Severity")]

H, p = kruskal(*groups)
print(f"Kruskal-Wallis: H = {H:.2f}, p = {p:.3g}")

# 2) Severity as an ordered scale: does the loss increase with severity?
# plot
plt.scatter(sev["Severity"], sev["Property loss usd"], alpha=0.3)

x = sev["Severity"]
y = sev["Property loss usd"]

coef = np.polyfit(x, y, 1)
plt.plot(x, coef[0] * x + coef[1], linestyle="--")

plt.xlabel("Severity")
plt.ylabel("Property loss (USD)")
plt.xticks([1, 2, 3, 4])
plt.show()

# Spearman correlation test
rho, p = spearmanr(sev["Severity"], sev["Property loss usd"])
print(f"Spearman: rho = {rho:.3f}, p = {p:.3g}")